In [8]:
import numpy as np
import h5py
import os

In [9]:
# Sampling + tensor shape
FS = 100                 # Hz, matches hardware ODR
T_MAX = 4500             # samples = 9 s @ 500 Hz
N_CHANNELS = 30          # 5 IMUs x (accel xyz + gyro xyz)
N_IMUS = 5
ACCEL_AXES = [0, 1, 2]   # relative to each IMU's 6-channel block
GYRO_AXES = [3, 4, 5]

# Per-class parameters. Index = class label.
CLASS_PARAMS = [
    {"accel_std": 0.3, "gyro_std": 0.2, "length": 1500, "freq_hz": 1.0},
    {"accel_std": 0.8, "gyro_std": 0.6, "length": 2500, "freq_hz": 2.0},
    {"accel_std": 1.5, "gyro_std": 1.2, "length": 3800, "freq_hz": 3.0},
]

NOISE_STD = 0.1          # shared Gaussian noise floor across all classes
SINE_AMPLITUDE = 0.5     # amplitude of the class-specific sinusoidal component

In [10]:
"""
Synthetic IMU data generator for APEX pipeline validation.

Purpose: produce a 3-class dataset where the class-separating features are
known in advance, so that end-to-end pipeline correctness (ingestion -> model
-> evaluation) can be verified before real climbing data exists.

Class design (difficulty ordinal: 0=easy, 1=medium, 2=hard):
    - accel std       : 0.3  / 0.8  / 1.5
    - gyro std        : 0.2  / 0.6  / 1.2
    - attempt length  : 1500 / 2500 / 3800 samples (rest zero-padded to 4500)
    - dominant freq   : 1 Hz / 2 Hz / 3 Hz sinusoidal component

All 30 channels (5 IMUs x 6 axes) share the per-class statistics. Channels 0-2
of each IMU are accelerometer-like; channels 3-5 are gyro-like. Same Gaussian
noise floor (sigma=0.1) on every class, added before zero-padding so pads stay
exactly zero.
"""


def _generate_one_attempt(params, rng):
    """Generate a single (T_MAX, 30) attempt for the given class params."""
    length = params["length"]
    accel_std = params["accel_std"]
    gyro_std = params["gyro_std"]
    freq_hz = params["freq_hz"]

    # Start with zeros so the pad region (length:T_MAX) stays exactly zero.
    x = np.zeros((T_MAX, N_CHANNELS), dtype=np.float32)

    # Time vector for the active portion of the attempt.
    t = np.arange(length) / FS  # seconds

    # Class-specific sinusoidal component shared across channels but with a
    # random phase per channel so the network sees coordinated-but-not-identical
    # oscillation across IMUs.
    phases = rng.uniform(0, 2 * np.pi, size=N_CHANNELS).astype(np.float32)
    sine = SINE_AMPLITUDE * np.sin(
        2 * np.pi * freq_hz * t[:, None] + phases[None, :]
    ).astype(np.float32)

    # Structured random walk in accel channels scaled to accel_std, plus the
    # same idea scaled to gyro_std for gyro channels. This gives the variance
    # signal its raw magnitude.
    for imu in range(N_IMUS):
        base = imu * 6
        for axis in ACCEL_AXES:
            ch = base + axis
            x[:length, ch] = rng.normal(0.0, accel_std, size=length)
        for axis in GYRO_AXES:
            ch = base + axis
            x[:length, ch] = rng.normal(0.0, gyro_std, size=length)

    # Overlay the class-specific sinusoid on the active region only.
    x[:length, :] += sine

    # Gaussian noise floor, same sigma for all classes, applied only to the
    # active region. The zero-padded tail remains identically zero so the
    # transition index is an unambiguous duration cue.
    x[:length, :] += rng.normal(0.0, NOISE_STD, size=(length, N_CHANNELS)).astype(np.float32)

    return x

In [11]:
def generate_synthetic_imu(n_per_class, seed=42):
    """
    Generate a synthetic 3-class IMU dataset for APEX pipeline validation.

    Parameters
    ----------
    n_per_class : int
        Number of attempts per class. Total samples = 3 * n_per_class.
    seed : int
        RNG seed for reproducibility.

    Returns
    -------
    X : np.ndarray, shape (3*n_per_class, 4500, 30), dtype float32
    y : np.ndarray, shape (3*n_per_class,),           dtype int64
        Class labels: 0=easy, 1=medium, 2=hard.
    """
    rng = np.random.default_rng(seed)

    n_total = 3 * n_per_class
    X = np.zeros((n_total, T_MAX, N_CHANNELS), dtype=np.float32)
    y = np.zeros(n_total, dtype=np.int64)

    idx = 0
    for class_label in range(3):
        params = CLASS_PARAMS[class_label]
        for _ in range(n_per_class):
            X[idx] = _generate_one_attempt(params, rng)
            y[idx] = class_label
            idx += 1

    # Shuffle so class order is not an artifact in any downstream split.
    perm = rng.permutation(n_total)
    return X[perm], y[perm]

In [12]:
X, y = generate_synthetic_imu(n_per_class=50, seed=42)
print(f"X shape: {X.shape}, dtype: {X.dtype}")
print(f"y shape: {y.shape}, class counts: {np.bincount(y)}\n")

print("Per-class verification (should be monotonically increasing):")
print(f"{'class':<8}{'accel_std':<14}{'gyro_std':<14}{'nonzero_len':<14}{'duration_s':<14}")

accel_chs = np.array([imu * 6 + ax for imu in range(N_IMUS) for ax in ACCEL_AXES])
gyro_chs  = np.array([imu * 6 + ax for imu in range(N_IMUS) for ax in GYRO_AXES])

for c in range(3):
    Xc = X[y == c]

    # Nonzero length per attempt
    lengths = np.array([np.any(attempt != 0, axis=1).sum() for attempt in Xc])

    # Per-attempt std over the active region, using np.ix_ for clean indexing
    accel_stds = []
    gyro_stds  = []
    for i in range(len(Xc)):
        active = Xc[i, :lengths[i], :]                # (L, 30)
        accel_stds.append(active[:, accel_chs].std())
        gyro_stds.append(active[:, gyro_chs].std())

    mean_len = lengths.mean()
    duration_s = mean_len / FS
    print(f"{c:<8}{np.mean(accel_stds):<14.3f}{np.mean(gyro_stds):<14.3f}{mean_len:<14.0f}{duration_s:<14.1f}")

X shape: (150, 4500, 30), dtype: float32
y shape: (150,), class counts: [50 50 50]

Per-class verification (should be monotonically increasing):
class   accel_std     gyro_std      nonzero_len   duration_s    
0       0.475         0.418         1500          15.0          
1       0.880         0.704         2500          25.0          
2       1.544         1.255         3800          38.0          


### Write the dataset to HDF5 format

Single self-describing file with all metadata as HDF5 attrs. The model notebook loader reads this directly — no CSV detour. When real data arrives in W5, the writer here is replaced but the reader in `model.ipynb` stays identical; you just swap `synthetic_v1.h5` for `real_v1.h5`.

**Schema:**
- `/X`: float32, shape `(N, 4500, 30)` — raw IMU tensor
- `/y`: int64, shape `(N,)` — class labels 0/1/2
- root attrs: `fs`, `n_channels`, `t_max`, `class_names`, `generator_seed`, `generator_version`, `created_at`

In [13]:
from datetime import datetime

# Generate the full dataset for training. 500/class = 1500 total attempts,
# enough to stress-test the pipeline without eating 800+ MB of RAM per load.
# don't confound pipeline debugging.
N_PER_CLASS = 200
OUTPUT_PATH = 'dataset/synthetic_v1.h5'
GENERATOR_VERSION = 'synthetic_v1'
SEED = 42

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

X, y = generate_synthetic_imu(n_per_class=N_PER_CLASS, seed=SEED)
print(f"Generated: X={X.shape} ({X.nbytes / 1e6:.1f} MB), y={y.shape}")

with h5py.File(OUTPUT_PATH, 'w') as f:
    # Datasets with gzip compression — synthetic data compresses well,
    # and real data will have enough structure to benefit too.
    f.create_dataset('X', data=X, compression='gzip', compression_opts=4)
    f.create_dataset('y', data=y, compression='gzip', compression_opts=4)

    # Root attrs — everything needed to reconstruct context
    f.attrs['fs'] = FS
    f.attrs['n_channels'] = N_CHANNELS
    f.attrs['n_imus'] = N_IMUS
    f.attrs['t_max'] = T_MAX
    f.attrs['class_names'] = np.array(['easy', 'medium', 'hard'], dtype='S')
    f.attrs['generator_seed'] = SEED
    f.attrs['generator_version'] = GENERATOR_VERSION
    f.attrs['created_at'] = datetime.now().isoformat()
    f.attrs['n_per_class'] = N_PER_CLASS

    # Channel layout attrs — record which channels are accel vs gyro so the
    # model notebook (or any downstream code) doesn't have to hardcode it
    f.attrs['accel_channels'] = np.array(
        [imu * 6 + ax for imu in range(N_IMUS) for ax in ACCEL_AXES], dtype=np.int32)
    f.attrs['gyro_channels'] = np.array(
        [imu * 6 + ax for imu in range(N_IMUS) for ax in GYRO_AXES], dtype=np.int32)

file_size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Wrote {OUTPUT_PATH} ({file_size_mb:.1f} MB on disk)")

Generated: X=(600, 4500, 30) (324.0 MB), y=(600,)
Wrote dataset/synthetic_v1.h5 (175.0 MB on disk)


### Validate Save

In [14]:
with h5py.File(OUTPUT_PATH, 'r') as f:
    X_loaded = f['X'][:]
    y_loaded = f['y'][:]
    print("Datasets:")
    print(f"  X: shape={X_loaded.shape}, dtype={X_loaded.dtype}")
    print(f"  y: shape={y_loaded.shape}, dtype={y_loaded.dtype}, counts={np.bincount(y_loaded)}")
    print("\nAttrs:")
    for key in f.attrs:
        val = f.attrs[key]
        if isinstance(val, np.ndarray) and val.dtype.kind == 'S':
            val = [s.decode() for s in val]
        print(f"  {key}: {val}")

# Round-trip integrity check
assert np.array_equal(X, X_loaded), "X mismatch after round-trip"
assert np.array_equal(y, y_loaded), "y mismatch after round-trip"
print("\nRound-trip OK: loaded data matches in-memory arrays exactly.")

Datasets:
  X: shape=(600, 4500, 30), dtype=float32
  y: shape=(600,), dtype=int64, counts=[200 200 200]

Attrs:
  accel_channels: [ 0  1  2  6  7  8 12 13 14 18 19 20 24 25 26]
  class_names: ['easy', 'medium', 'hard']
  created_at: 2026-04-09T01:39:40.998734
  fs: 100
  generator_seed: 42
  generator_version: synthetic_v1
  gyro_channels: [ 3  4  5  9 10 11 15 16 17 21 22 23 27 28 29]
  n_channels: 30
  n_imus: 5
  n_per_class: 200
  t_max: 4500

Round-trip OK: loaded data matches in-memory arrays exactly.
